# ACCESS-AIS3 -- Stage 1: HO thermal steady-state

**Post-inversion extensions context:**

Standard practice after a friction inversion: higher-order thermal spin-up (SSA has
no vertical shear physics, so temperature/rheology can't be solved self-consistently
-- ais_0.1_param.py's surface-temperature-as-proxy approach is a known, flagged
caveat), a friction re-inversion under that higher-order physics (SSA-tuned friction
isn't valid once vertical shear resistance is added), ocean melt-rate calibration, a
short post-inversion relaxation, and a historical run tuned against observed dH/dt.

Stage 1 (`ho_thermal_steadystate`) is implemented and ready to test. Stages 2-5 are
scaffolds: correct model loading / solver setup / save-submit structure, each with an
explicit TODO marking the science decision that still needs iteration. Do not treat
their output as validated the way `AIS3_inverted.nc` / `AIS3_relaxed.nc` are once
stage 1 has actually been run and checked.

See `docs/inversion_worklog.md` and the originating plan for the full investigation.


## Imports & helper functions

In [ ]:
import pyissm
import ccdtools as ccdtools
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
import pandas as pd
import xarray as xr
import os


def friction_law_info(md):
    """Return (control_parameter, field_attr, min_bound, max_bound) for md's friction law.

    Schoof (regularized Coulomb) inverts 'FrictionC' (field md.friction.C); Budd/Weertman (the
    'default' class) inverts 'FrictionCoefficient' (field md.friction.coefficient). The saved
    friction class (set by friction_law in ais_0.1_param.py) is the single source of truth.
    """
    if type(md.friction).__name__ == 'default':   # Budd / Weertman power law
        # VALIDATED bounds [0.05, 900] for p=q=1 (grounded RMSE 61.4 unregularised / 60.4 with
        # cf501=0.0001, see ais_0.1_param.py). The earlier [0.1, 10] bounds were tuned for the
        # superseded p=q=3 law (u ~ C^-6, `friconly_nfix`, RMSE 98.9) -- under p=q=1 (u ~ C^-2)
        # fast ice needs a much larger C for the same resisting stress, and that ceiling pinned
        # 41% of the domain at C=10 the first time p=1 was tried with it.
        return 'FrictionCoefficient', 'coefficient', 0.05, 900
    # Schoof (regularized Coulomb): tested directly for the Siple Coast trunk deficit and ruled
    # out on physics grounds, not just numerics -- see docs/inversion_worklog.md section 5.4.
    # Coupled-domain adjoint inversion of FrictionC is unstable near the Coulomb cap regardless
    # of solver settings, and a forward-only sweep across the full documented Cmax range
    # (0.17-0.84) left the Siple Coast trunk ratio completely unchanged from Budd's. Kept here
    # only so friction_law='schoof' remains loadable; not the recommended path.
    return 'FrictionC', 'C', 0.05, 250 ** 2        # Schoof (regularized Coulomb)


def extract_friction_inversion_domain(md):
    """Extract the friction-inversion subdomain with a floating/grounded ice-front boundary condition.

    Extracts ALL ice (ice_levelset_elements < 1, includes the ice-front elements), so the new
    mesh boundary coincides exactly with the true, contiguous ice margin -- not an arbitrary
    internal cut. extract() imposes Dirichlet (observed velocity) on every new boundary node by
    default (see Model.py: "Boundary conditions: Dirichlets on new boundary"); this is reverted
    to Neumann (NaN spc) at boundary nodes classified as floating (ocean_levelset < 0), i.e. true
    ice-shelf calving fronts, where the natural ocean-pressure BC is physically correct. Boundary
    nodes classified as grounded (ocean_levelset >= 0) -- both marine-terminating (bed below sea
    level, no shelf) and true land-terminating (bed above sea level, no ocean to push back
    against) -- keep extract()'s default Dirichlet, since Neumann has no obvious physical meaning
    there. Classification is per-vertex (mds.mask.ocean_levelset), not per-element, so it follows
    the true ice-front geometry exactly with no fragmentation.

    A prior version anchored only the Ronne-Filchner/Ross fronts (the two largest floating
    regions, found to blow up under pure Neumann at low friction coefficient) and left everything
    else -- including land-terminating margins -- as Neumann. That fixed Ronne-Filchner/Ross but
    left land-terminating margins with a physically meaningless Neumann BC, which was the actual
    cause of a ~1e10 m/yr blowup at coeff=1 (confirmed: switching those margins to Dirichlet here
    brought coeff=1 down to ~1.8e7 m/yr).
    """
    ice_levelset_elements = pyissm.tools.interp.vertex_to_element(md, md.mask.ice_levelset)
    mds = md.extract(ice_levelset_elements < 1)

    bnd = mds.mesh.vertexonboundary.astype(bool)
    ocean_ls = np.asarray(mds.mask.ocean_levelset).ravel()
    floating_bnd = bnd & (ocean_ls < 0)

    mds.stressbalance.spcvx[floating_bnd] = np.nan
    mds.stressbalance.spcvy[floating_bnd] = np.nan
    mds.stressbalance.spcvz[floating_bnd] = np.nan
    mds.mask.ice_levelset[floating_bnd] = 0

    return mds


def load_shelf_rheology_B():
    """Load the validated floating-shelf rheology inversion result (extractedvertices, B).

    execution_newB_rheology/run_001_1_10_1e-17 (2026-07-20) supersedes
    models/AIS3_ssa_rheology_floating_inv_lcurve/run_004_1_10_1e-17 (2026-06-30, the
    `rheology_lcurve_run` config value): same regularisation point (cf101=1, cf103=10,
    cf502=1e-17), recomputed later in this project after several geometry/N-flooring fixes
    were developed (100m thickness floor, N re-flooring against it, etc.) -- run_004 predates
    those fixes. This is the actual source the validated grounded-RMSE-60.4 friction result
    was warm-started from; loading the stale run_004 instead was found (via a direct A/B
    test) to reproduce RMSE ~114, not ~60 -- see docs/inversion_worklog.md. Not yet promoted
    into the canonical models/AIS3_ssa_rheology_floating_inv_lcurve/ directory, so this loads
    it from its original ad-hoc execution directory via solve(load_only=True) instead of
    io.load_model().
    """
    _cl = pyissm.model.classes.cluster.gadi()
    _cl.codepath = os.environ['ISSM_DIR'] + '/bin'
    _cl.executionpath = '/g/data/au88/jh7060/ACCESS-AIS3/execution_newB_rheology'
    _cl.login = 'jh7060'; _cl.project = 'au88'; _cl.storage = 'gdata/au88'

    mshelf = pyissm.model.io.load_model(f'{model_dir}/AIS3_param.nc')
    mshelf.mask.ice_levelset = pyissm.model.param.kill_icebergs(mshelf)
    sel = (mshelf.mask.ocean_levelset < 0) & (mshelf.mask.ice_levelset < 0)
    mshelf = mshelf.extract(sel)
    mshelf.cluster = _cl
    mshelf.settings.waitonlock = 0
    mshelf.inversion.iscontrol = 0
    mshelf.miscellaneous.name = 'run_001_1_10_1e-17'
    mr = pyissm.model.execute.solve(mshelf, 'Stressbalance', load_only = True, runtime_name = False, check_consistency = False)
    return np.asarray(mr.mesh.extractedvertices).ravel(), np.asarray(mr.results.StressbalanceSolution.MaterialsRheologyBbar).ravel()

## Configure options

In [ ]:
## ------------------------------------
## Configure options
## ------------------------------------

# Change directory to gdata to prevent storage limits in $HOME
os.chdir('/g/data/au88/jh7060/ACCESS-AIS3/')
os.environ['ISSM_DIR'] = '/g/data/vk83/apps/spack/1.1/release/linux-x86_64/issm-git.2026.05.18_2026.05.18-kgta35igm37z4qnqnul7rcmgx2inftqd'

# Should plots be generated?
plot = True
diagnostics = True
save = True
inversion_sensitivity = False

# Define execution directory
execution_dir = '/g/data/au88/jh7060/ACCESS-AIS3/execution'

# Define location to save final models
model_dir = '/g/data/au88/jh7060/ACCESS-AIS3/models'

# Define domain_file
domain_file = ('/g/data/au88/jh7060/ACCESS-AIS3/assets/ais_domain.exp')

# Define param_file
param_file = ('/g/data/au88/jh7060/ACCESS-AIS3/config/ais_0.1_param.py')

# Define cluster requirements
cluster = pyissm.model.classes.cluster.gadi()
cluster.codepath = os.environ['ISSM_DIR']+'/bin'
cluster.executionpath = execution_dir
cluster.storage = 'gdata/au88+gdata/vk83'
cluster.moduleuse = ['/g/data/vk83/modules/']
cluster.moduleload = ['access-issm_ad/2026.05.0']  # was access-issm/2025.11.0: executing a
# 2026.05.18 binary under a 2025.11.0 module load -- a stale-module mismatch caught and fixed
# across every scratchpad script this session; production had not been updated to match.
# np/memory: 32 cores / 100GB is under-provisioned -- this mesh needs ~130GB minimum even at
# 32 ranks (see docs/inversion_worklog.md section 8), which is the likely real cause of the
# OOM history noted below on the maxsteps line, not maxsteps itself. 48 cores / 190GB is the
# configuration validated as SU-optimal this session (>96 cores was actively worse).
cluster.np = 48
cluster.memory = 190
cluster.time = 60*48
cluster.login = 'jh7060'
cluster.project = 'au88'


all_steps = [
    'process_domain',
    'mesh',
    'param',
    'ssa_rheology_floating_inv_sensit',
    'ssa_rheology_floating_inv_lcurve',
    # 'ssa_rheology_floating_inv',
    'ssa_friction_forward_check',
    'ssa_friction_forward_check_budd',
    'ssa_friction_inv_sensit',
    'ssa_friction_inv_lcurve',
    'ssa_friction_inv_reg_lcurve',
    'ssa_inverted_solve',
    'ssa_relaxation',
    'ho_thermal_steadystate',
    'ho_friction_inv',
    'melt_gamma_tuning',
    'ho_relaxation',
    'historical_dhdt_tuning',
]

# Define steps to run (this notebook is scoped to this group)
steps = ['ho_thermal_steadystate']

## Chosen inversion runs (update after inspecting sensit / lcurve diagnostics)

In [ ]:
## ------------------------------------
## Chosen inversion runs (update after inspecting sensit / lcurve diagnostics)
## ------------------------------------
# Floating-ice rheology B field taken from the rheology L-curve (cf502 regularisation).
rheology_lcurve_run = 'run_004_1_10_1e-17'

# Preferred 101/103 cost-function coefficients for the friction inversion.
# The cf101=1000/cf103=0.1 choice below (run_021, vel_rmse=960.5) came from a sensit sweep run
# against the C_init=10 dead-zone bug (see ais_0.1_param.py): with u ~ C^-6 and the model stuck
# at zero velocity everywhere, that sweep's vel_rmse was never measuring model skill (it was
# ~equal to RMS(v_obs) itself, i.e. the null model). Every cell in that grid is void.
# VALIDATED instead (grounded RMSE 98.9, `friconly_nfix`): cf101=10, cf103=100 -- log-weighted,
# so the slow interior (which absolute weighting like 1000/0.1 effectively ignores) contributes
# to the fit. 10/100 was carried through every successful run this pipeline is based on.
friction_cf101 = 10
friction_cf103 = 100

# Mirrors friction_law in ais_0.1_param.py -- that flag only lives inside ais_0.1_param.py's
# own exec-scope (set on md.friction when 'param' in steps calls parameterize(), see below),
# so it isn't otherwise visible here at module load time where friction_lcurve_run (needed by
# ssa_inverted_solve) is defined. Keep this in sync with ais_0.1_param.py by hand.
friction_law = 'schoof'  # 'schoof' or 'budd' -- must match ais_0.1_param.py

# Effective-pressure source for the friction law. coupling=2 (ISSM internal "uniform sheet"
# hydrology, clamped >= 0) matched or beat coupling=3 (Ehrenfeucht dataset + manual N floor) in
# the earlier *uniform-coefficient forward-check* sweep -- but the full floating/grounded-BC
# inversion sensit sweep told a different story: coupling=2 has a specific, severe pathology at
# certain coefficient cells (cf101=cf103=10 and cf101=cf103=1000 both spiked to vel_rmse~13,100,
# ~13x every other cell) that the simpler forward-check never happened to probe. coupling=3 was
# clean and outlier-free across the entire 25-cell grid (vel_rmse 962-1385, no spikes). Reverted
# to coupling=3 on the strength of that full-grid evidence. AIS3_param.nc itself is also built
# with friction_coupling=3 (see ais_0.1_param.py), so this now matches the param-file default.
friction_coupling = 3

# m1qn3's relative gradient-norm stopping tolerance (default 1e-4). ROOT CAUSE of every earlier
# p=q=1 "convergence" that silently never fit anything: under p=1's much gentler cost-function
# landscape than p=3's, ||g(X)||/||g(X0)|| falls below 1e-4 by iteration ~16, while the cost is
# still falling fast (not flattening) and the fit is nowhere near done -- a false stop, not a
# real one. Tightened well below anything that can trigger this early, so every successful run
# in this pipeline instead stops on dxmin (step-size), the genuine convergence criterion.
friction_inv_gttol = 1e-8

# Dirichlet-pin two known-unstable regions (Institute/Moller Ice Stream band + an isolated
# cluster near x~350km,y~-1933km) to observed velocity during the friction inversion, instead
# of leaving them free -- mirrors Felicity's own constrain_Budd.exp/constrain_Schoof.exp
# pattern (runme.m: Inversion_Friction_Budd/Schoof). Built from diag_schoof_blowup_v2.py's
# worst-25 grounded vertices (all sat at the 100m thickness floor with driving stress
# exceeding Cmax*N under Schoof -- see docs/inversion_worklog.md). Root cause of that specific
# blowup turned out to be the forced-Newton solver setting (isnewton=2), not geometry, so this
# flag is OFF by default until an actual A/B test shows it changes anything for Budd -- see
# ssa_friction_inv_reg_lcurve below.
use_constrain_regions = False
constrain_exp_file = '/g/data/au88/jh7060/ACCESS-AIS3/assets/constrain_Budd.exp'

# Grounded-ice friction C field, in two stages (see ssa_friction_inv_lcurve /
# ssa_friction_inv_reg_lcurve below):
#   1. `friction_baseline_run` -- the UNREGULARISED (cf501 effectively off) p=q=1 baseline,
#      grounded RMSE 61.4, used only as the warm-start for stage 2 below (its own C field is
#      usable but ~5x rougher, C-field roughness 0.82 vs the p=q=3 baseline's 0.17).
#   2. `friction_lcurve_run` -- warm-started from (1), light DragCoefficientAbsGradient
#      regularisation (cf501). cf501=0.0001 is the validated corner: it drops the C-field
#      roughness to 0.18 (matching p=q=3) while the RMSE *improves* further, to 60.4 -- not a
#      tradeoff, both axes move the same direction. This is the field `ssa_inverted_solve` uses.
#
# The Budd naming pattern above (run_001_{cf101}_{cf103}_{cf501}) is specific to the Budd
# L-curve sweep; it does not apply to the Schoof m1qn3 continuation run (different control
# parameter, different script, not a cf501 grid point), so this is branched on friction_law
# rather than reused. See friction_law in ais_0.1_param.py for the full rationale for the
# current Schoof choice; ssa_friction_inv_reg_lcurve has never actually been run for Schoof --
# this points at the scratchpad-run tight-restol result saved into this same directory
# structure by finalize_schoof_friction_result.py, not a production-pipeline output.
if friction_law == 'schoof':
    friction_baseline_run = 'schoof_m1qn3_tightrestol_cmax2.0'
    friction_lcurve_run = 'schoof_m1qn3_tightrestol_cmax2.0'
else:
    friction_baseline_run = f'run_001_{friction_cf101}_{friction_cf103}_1e-08'
    friction_lcurve_run = f'run_001_{friction_cf101}_{friction_cf103}_0.0001'

## Initialise data catalog

In [ ]:
## ------------------------------------
## Initialise Data Catalog
## ------------------------------------
catalog = ccdtools.catalog.DataCatalog()
bedmachine_data = catalog.load_dataset('measures_bedmachine_antarctica', version = 'v3')
velocity_data = catalog.load_dataset('measures_insar_based_antarctica_ice_velocity_map', version = 'v2')
measures_coastline = catalog.load_dataset('measures_antarctic_boundaries', subdataset = 'coastline')

## Stage 1: Higher-order (HO) velocity + single-shot thermal solve

In [ ]:
# Solves HO velocity once (frozen), then a single non-coupled thermal solve against that
# frozen field -- replacing ais_0.1_param.py's surface-temperature-as-proxy rheology_B
# with a real depth-resolved one. Originally attempted as MOLHO + a coupled
# SteadystateSolution Picard loop (velocity<->thermal iterated to convergence); MOLHO's
# reduced basal+shear velocity representation and the coupled loop's slow/unstable
# convergence on fast, warm-bedded basins (see PIG/Thwaites small-region test) both proved
# unworkable. Switched to full HO (matches Felicity's actual validated Thermal step,
# runme.m:772-836) and the decoupled solve-once structure instead. See the isenthalpy=0
# comment below for the other major deviation from the original plan.
if 'ho_thermal_steadystate' in steps:

    print("-------------------------------------------------------------")
    print(f" HIGHER-ORDER (HO) VELOCITY + SINGLE-SHOT THERMAL SOLVE"      )
    print("-------------------------------------------------------------")

    print(f"-- Loading inverted model...")
    md = pyissm.model.io.load_model(f'{model_dir}/AIS3_inverted.nc')

    # BUGFIX (found via the PIG/Thwaites small-region test): initialization.waterfraction
    # and .watercolumn both default to a bare scalar NaN (pyissm/model/classes/
    # initialization.py:85,89) -- same NaN-default-doesn't-survive-an-operation pattern hit
    # repeatedly earlier in this pipeline (masstransport.spcthickness,
    # basalforcings.*_melting_rate). Model.extrude() calls mesh._project_3d() on these
    # fields to replicate them across layers (initialization.py:142-143), but a bare scalar
    # can't be broadcast correctly there -- it comes out still shape (1,) instead of
    # (numberofvertices3d,), tripping the post-extrude consistency check. Zero (no
    # subglacial water) is the standard placeholder absent real hydrology forcing, same
    # status as the SMB/melt placeholders in ssa_relaxation.
    md.initialization.waterfraction = np.zeros(md.mesh.numberofvertices)
    md.initialization.watercolumn = np.zeros(md.mesh.numberofvertices)

    # BUGFIX (found via the PIG/Thwaites small-region test): same NaN-default gap as
    # ssa_relaxation hit -- basalforcings.{groundedice,floatingice}_melting_rate default to
    # bare scalar NaN and were never exercised on this model before (AIS3_inverted.nc only
    # ever went through a Stressbalance solve). This is a marshalling-time crash
    # ("FetchDataToInput ... not found in binary file"), not caught by the Python
    # consistency check, so it only surfaces once the job actually submits. Zero basal melt
    # is the same placeholder used in ssa_relaxation, pending melt_gamma_tuning.
    md.basalforcings.groundedice_melting_rate = np.zeros(md.mesh.numberofvertices)
    md.basalforcings.floatingice_melting_rate = np.zeros(md.mesh.numberofvertices)

    print('-- Removing icebergs from ice levelset...')
    md.mask.ice_levelset = pyissm.model.param.kill_icebergs(md)

    print(f"-- Extruding mesh (15 layers)...")
    md = md.extrude(num_layers = 15, extrusion_exponent = 1.3)

    print(f"-- Setting flow equation to HO...")
    # SWITCHED from MOLHO to full HO -- matches Felicity's actual validated Thermal step
    # (runme.m:800, `setflowequation(md, 'HO', 'all')`) exactly. The PIG/Thwaites
    # small-region test found MOLHO's reduced basal+shear velocity representation,
    # reconstructed into a full 3D field to drive thermal advection, produced an
    # oscillating/ill-conditioned thermal solve regardless of mesh resolution,
    # stabilization scheme, or residue threshold; full HO (independent velocity DOF at
    # every layer, no reconstruction needed) converged far more cleanly. No MOLHO-specific
    # BC split (set_molho_bc) needed for plain HO.
    md = pyissm.model.param.set_flow_equation(md, HO = 'all')

    print(f"-- Priming a depth-varying initial temperature profile...")
    # ROOT CAUSE found via the PIG/Thwaites small-region test: initialization._extrude()
    # calls mesh._project_3d(..., type='node') with the default layer=0, which REPLICATES
    # the 2D surface air temperature identically at every one of the 15 vertical layers
    # (pyissm/model/mesh.py:2607-2609, layer==0 branch). Real ice columns warm
    # substantially toward the bed (geothermal flux + strain heating), often approaching
    # pressure melting -- starting every layer at cold surface temperature forces the
    # nonlinear solver to build the entire vertical thermal structure from a flat,
    # far-from-equilibrium guess. Priming a linear profile from surface temp (top) to
    # just-below-pressure-melting (bed) is standard ice-sheet thermal-spinup practice.
    _bed3d = np.asarray(md.geometry.bed).ravel()
    _surf3d = np.asarray(md.geometry.surface).ravel()
    _z3d = np.asarray(md.mesh.z).ravel()
    _thickness3d = np.maximum(_surf3d - _bed3d, 1.0)
    _depth_frac = np.clip((_surf3d - _z3d) / _thickness3d, 0.0, 1.0)  # 0 at surface, 1 at bed
    _pressure = md.materials.rho_ice * md.constants.g * np.maximum(_surf3d - _z3d, 0.0)
    _Tpmp = md.materials.meltingpoint - md.materials.beta * _pressure
    _basal_target = np.minimum(_Tpmp, 273.15) - 1.0
    _surf_temp3d = np.asarray(md.initialization.temperature).ravel()
    md.initialization.temperature = _surf_temp3d + _depth_frac * (_basal_target - _surf_temp3d)
    md.initialization.temperature = np.clip(md.initialization.temperature, 220.0, 273.15)

    # Felicity's stabilization trick (runme.m:823-824): pin non-ice vertices to a fixed
    # temperature -- non-ice extruded columns are geometrically degenerate (near-zero real
    # ice thickness there), so their thermal equation is ill-posed without a Dirichlet
    # constraint. CONFIRMED NECESSARY empirically: removing it let the solve finish
    # "cleanly" but produced min temperature -8,399,632 K. Also initializing temperature
    # at those same nodes to match the pin target, so the constraint starts consistent
    # with the initial guess rather than jumping to it (the mismatch was the likely cause
    # of an early large first-solve residual in testing).
    _non_ice = np.asarray(md.mask.ice_levelset).ravel() > 0
    _temp = np.asarray(md.initialization.temperature).ravel().copy()
    _temp[_non_ice] = 250.0
    md.initialization.temperature = _temp

    md.timestepping.time_step = 0
    md.inversion.iscontrol = 0
    md.verbose.solution = 1

    print(f"-- Assigning cluster and updating settings...")
    # BUGFIX: the shared `cluster` object (190GB/normal queue) is correctly tuned for
    # this pipeline's 2D SSA problems, but the full-continental run crashed here with
    # Exit Status 137 (SIGKILL, OOM) at 180.8GB used / 190GB requested -- a 15-layer
    # extruded HO mesh (~23.8M 3D nodes, full independent velocity DOF per layer, unlike
    # MOLHO's reduced basal+shear representation) needs far more memory than the 2D
    # problems this cluster config was tuned for. Using a local override (hugemem queue,
    # ~2900GB/node) rather than changing the shared object other steps rely on.
    md.cluster = pyissm.model.classes.cluster.gadi()
    md.cluster.codepath = cluster.codepath
    md.cluster.executionpath = cluster.executionpath
    md.cluster.storage = cluster.storage
    md.cluster.moduleuse = cluster.moduleuse
    md.cluster.moduleload = cluster.moduleload
    md.cluster.login = cluster.login
    md.cluster.project = cluster.project
    md.cluster.queue = 'hugemem'
    md.cluster.np = 48
    # BUGFIX: 2900GB exceeded hugemem's actual per-node cap -- PBS rejected the qsub
    # outright ("requested more memory per node (2.83TB) than the nodes in queue hugemem
    # can provide"), and pyissm's solve() didn't surface that failure, so the launcher sat
    # forever "waiting for lock file" on a job that was never created. Confirmed via direct
    # qsub testing that 1470-1500GB is accepted; using 1450GB for headroom below whatever
    # the exact ceiling is.
    md.cluster.memory = 1450
    md.cluster.time = 60 * 48
    md.settings.solver_residue_threshold = 1e-3

    # Step 1: solve HO velocity ONCE, then freeze it -- mirrors Felicity's structure
    # exactly (solve HO, save, load, solve thermal against the frozen field), NOT a
    # coupled SteadystateSolution Picard loop. The coupled loop was tried first and made
    # every attempt converge extremely slowly (strong two-way velocity<->temperature
    # feedback on fast, warm-bedded basins) before failing outright; the decoupled
    # structure sidesteps that feedback entirely. maxiter=100 (not Felicity's maxiter=5)
    # after testing showed 5 was too rough for our SSA-inverted friction field's known
    # residual noise -- an under-converged velocity here destabilized the thermal solve.
    print(f"-- Solving HO stress balance (frozen velocity for the thermal solve)...")
    md.stressbalance.maxiter = 100
    md.miscellaneous.name = 'AIS3_thermal_steadystate_velocity'
    md.settings.waitonlock = 1440  # minutes
    md = pyissm.model.execute.solve(md, 'stressbalance', load_only = False, runtime_name = False)

    if diagnostics:
        vel_check = np.asarray(md.results.StressbalanceSolution.Vel).ravel()
        print(f"   velocity solve done; max Vel = {np.nanmax(vel_check):.1f} m/yr")

    md.initialization.vx = md.results.StressbalanceSolution.Vx
    md.initialization.vy = md.results.StressbalanceSolution.Vy
    md.initialization.vz = md.results.StressbalanceSolution.Vz
    md.initialization.vel = md.results.StressbalanceSolution.Vel
    md.initialization.pressure = md.results.StressbalanceSolution.Pressure

    # Step 2: single (non-coupled) thermal solve against that frozen velocity field.
    print(f"-- Configuring and solving single (non-coupled) thermal solve...")
    # isenthalpy=0, NOT Felicity's isenthalpy=1: every attempt with the enthalpy (phase-
    # change) formulation failed -- residue either diverged outright or oscillated after
    # nearly clearing the threshold (0.10 vs 0.1), across every combination of mesh
    # resolution, HO vs MOLHO, stabilization scheme, residue threshold, and penalty_lock
    # tried. The plain temperature formulation converged cleanly on the first attempt
    # after switching. Known caveat: without enthalpy, temperature isn't capped at the
    # pressure-melting point -- ~2% of nodes came out slightly above it in testing (up to
    # +5.5K), fixed by the post-solve clip below rather than by the (much stiffer)
    # phase-change formulation.
    md.thermal.isenthalpy = 0
    md.thermal.maxiter = 100
    md.thermal.reltol = 0.05
    md.miscellaneous.name = 'AIS3_thermal_steadystate_thermal'
    md.settings.waitonlock = 1440  # minutes
    md = pyissm.model.execute.solve(md, 'thermal', load_only = False, runtime_name = False)

    # Post-solve cleanup, found necessary via the full continental run's own diagnostics
    # (not caught by the PIG/Thwaites small-region test): 7.2% of 2D columns came back
    # with catastrophically wrong temperature (<200K, many far below 0K), but 99.87% of
    # those (114,659 of 114,802) are NON-ICE columns -- the ones meant to be held at 250K
    # by the spctemperature pin. ISSM's spc constraints are penalty-based, not exact
    # elimination; the penalty evidently doesn't dominate strongly enough for a large
    # fraction of the ~114K scattered non-ice inclusions (rock outcrops, nunataks) spread
    # across the full continent, even though it held reliably for the much smaller/more
    # homogeneous non-ice population within the single PIG/Thwaites drainage basin. Since
    # non-ice temperature is scientifically meaningless anyway (no real ice column
    # exists there), forcibly overwriting it in Python is more robust than relying on the
    # solver's constraint to hold. Only 143 genuine ICE-covered columns (140 grounded, 3
    # floating -- 0.009% of the domain) were actually bad; velocity there is unremarkable
    # (mean 111.8 m/yr, below the 159.3 m/yr average for good grounded columns), ruling
    # out the fast-flow-instability mechanism suspected from PIG/Thwaites for this
    # residual population -- a floor clip is a defensible patch at this tiny scale.
    # Floor set to 200K, not 220K: diagnostics on the actual run showed nodes<200K
    # (1,606,647) and nodes<100K (1,605,850) are almost the same count (~800 apart) -- the
    # bad population is overwhelmingly catastrophic (deep negative, not gently cold), so a
    # 220K floor was needlessly warming genuine extreme-cold locations too. Real Antarctic
    # interior annual-mean temperatures (Dome A ~214.7K, Vostok ~217.9K) sit comfortably
    # above 200K, so this floor bounds true garbage without touching legitimate extremes.
    _T = np.asarray(md.results.ThermalSolution.Temperature).ravel()
    _non_ice_final = np.asarray(md.mask.ice_levelset).ravel() > 0
    _T[_non_ice_final] = 250.0
    _T = np.clip(_T, 200.0, 273.15)
    md.initialization.temperature = _T

    if diagnostics:
        T = md.initialization.temperature
        print(f"\nTHERMAL STEADY-STATE DIAGNOSTICS:")
        print(f"   Min temperature: {np.nanmin(T):.2f} K")
        print(f"   Max temperature: {np.nanmax(T):.2f} K")
        print(f"   Mean temperature: {np.nanmean(T):.2f} K")

    print(f"-- Recomputing rheology_B from the depth-resolved temperature...")
    md.materials.rheology_B = pyissm.tools.materials.cuffey(md.initialization.temperature)

    if save:
        print(f"\nSaving thermal steady-state model to {model_dir}/AIS3_thermal_steadystate.nc")
        pyissm.model.io.save_model(md, f'{model_dir}/AIS3_thermal_steadystate.nc')